# Failure-Aware Robustness Evaluation of TB Detection Models under Synthetic CXR Degradation

Reference implementation for the manuscript submitted to the *Egyptian Journal of
Radiology and Nuclear Medicine*.

**What this notebook does**

1. Builds a manifest of the Montgomery + Shenzhen chest X-ray sets (800 images).
2. Assigns non-repeated stratified five-fold cross-validation splits (70/10/20 within fold).
3. Fine-tunes ResNet-50, DenseNet-121 and MobileNetV2 end to end.
4. Selects one classification threshold per fold by Youden's J on the validation subset.
5. Applies five degradations at three severities **to the held-out test images only**.
6. Computes AUC, sensitivity, specificity, PPV, NPV and ECE, with paired bootstrap CIs.
7. Writes Tables 1, 3-7 and Supplementary Tables S1 and S2 from one set of predictions.
8. Runs consistency checks so every table reconciles with every other.

**Nothing here is hard-coded from the paper.** Every number is computed from the data.
Run the cells in order; Sections 1-7 take a few hours on one GPU, Sections 8-12 take seconds
once `test_predictions.csv` exists.

## 0. Configuration

Edit `DATA_ROOT` to point at your local copy of the two datasets. Everything else
matches the manuscript and should not be changed if you are reproducing the paper.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('data')          # expects data/montgomery/CXR_png and data/shenzhen/CXR_png
OUT       = Path('results')
(OUT / 'predictions').mkdir(parents=True, exist_ok=True)
(OUT / 'tables').mkdir(parents=True, exist_ok=True)

SEED        = 42
N_FOLDS     = 5
IMG_SIZE    = 224
BATCH_SIZE  = 16
LR          = 1e-4
MAX_EPOCHS  = 50
ES_PATIENCE = 10        # early stopping on validation AUC
LR_PATIENCE = 5         # ReduceLROnPlateau on validation loss
LR_FACTOR   = 0.1
N_BOOT      = 1000
N_BINS      = 10        # ECE, equal-width

MODELS = ['ResNet-50', 'DenseNet-121', 'MobileNetV2']

# Degradation grid: severity -> parameter, exactly as reported in the manuscript
SEVERITIES = {
    'Gaussian noise':     {'Mild': 0.01, 'Moderate': 0.02, 'Severe': 0.03},   # sigma
    'Motion blur':        {'Mild': 5,    'Moderate': 7,    'Severe': 9},      # kernel length
    'JPEG compression':   {'Mild': 90,   'Moderate': 75,   'Severe': 50},     # quality
    'Downsampling':       {'Mild': 0.75, 'Moderate': 0.67, 'Severe': 0.50},   # scale
    'Contrast reduction': {'Mild': 0.9,  'Moderate': 0.8,  'Severe': 0.7},    # gamma
}
CONDITIONS = ['Clean'] + list(SEVERITIES)

In [ ]:
import io, os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_seed(seed=SEED):
    """Single global seed for Python, NumPy and PyTorch (manuscript, Implementation Details)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '|', DEVICE)

## 1. Build the image manifest

Both NLM sets encode the label in the filename suffix: `_0` = no TB-consistent finding,
`_1` = TB-consistent finding. One radiograph per patient, so image-level and patient-level
splitting coincide.

Expected composition: Montgomery 138 (58 / 80), Shenzhen 662 (336 / 326),
combined 800 with 394 TB-positive (49.25%).

In [ ]:
def build_manifest(root=DATA_ROOT):
    rows = []
    for source, sub in [('Montgomery', 'montgomery/CXR_png'),
                        ('Shenzhen',   'shenzhen/CXR_png')]:
        folder = root / sub
        if not folder.exists():
            raise FileNotFoundError(f'{folder} not found - see data/README.md')
        for p in sorted(folder.glob('*.png')):
            stem = p.stem
            if stem.endswith('_1'):
                label = 1
            elif stem.endswith('_0'):
                label = 0
            else:
                continue
            rows.append({'image_id': stem, 'path': str(p),
                         'source': source, 'label': label})
    return pd.DataFrame(rows)


manifest = build_manifest()
print(manifest.groupby(['source', 'label']).size())
print('total', len(manifest), '| TB-positive', int(manifest.label.sum()),
      f'({100 * manifest.label.mean():.2f}%)')

assert len(manifest) == 800, 'expected 800 images'
assert manifest.label.sum() == 394, 'expected 394 TB-positive'

## 2. Fold assignment

Non-repeated stratified five-fold cross-validation: five mutually exclusive test folds
covering all 800 images. Within each fold the remainder is split 70/10/20 into
train / validation / test, stratified by label.

In [ ]:
def assign_folds(df, n_folds=N_FOLDS, seed=SEED):
    df = df.copy().reset_index(drop=True)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    df['fold'] = -1
    for k, (_, test_idx) in enumerate(skf.split(df, df.label), start=1):
        df.loc[test_idx, 'fold'] = k
    return df


def split_fold(df, fold, seed=SEED):
    """Return (train, val, test) for one fold. Test is the held-out fold;
    the remainder is split so the overall proportions are ~70/10/20."""
    test = df[df.fold == fold]
    rest = df[df.fold != fold]
    val_frac = 0.10 / 0.80          # 10% of the whole = 12.5% of the remaining 80%
    train, val = train_test_split(rest, test_size=val_frac,
                                  stratify=rest.label, random_state=seed)
    return train, val, test


manifest = assign_folds(manifest)
manifest[['image_id', 'source', 'label', 'fold']].to_csv(OUT / 'folds.csv', index=False)

for k in range(1, N_FOLDS + 1):
    tr, va, te = split_fold(manifest, k)
    print(f'fold {k}: train {len(tr)}  val {len(va)}  test {len(te)} '
          f'(TB-positive in test: {int(te.label.sum())})')

## 3. Degradations

Applied **after** resize and per-image min-max normalisation, to test images only,
on a float32 array in [0, 1].

| Degradation | Implementation notes |
|---|---|
| Gaussian noise | zero-mean, added in [0,1], result clipped to [0,1] |
| Motion blur | linear kernel of length k, horizontal, replicate padding |
| JPEG | encode to 8-bit greyscale and decode back, so the condition includes quantisation |
| Downsampling | bilinear down then bilinear back to 224, keeping network input size constant |
| Contrast reduction | gamma on the normalised image; endpoints preserved |

The noise realisation is drawn from a generator seeded per (fold, severity), so the same
realisation is used for every model at a given condition.

In [ ]:
import cv2


def deg_gaussian_noise(img, sigma, rng):
    return np.clip(img + rng.normal(0.0, sigma, img.shape).astype(np.float32), 0.0, 1.0)


def deg_motion_blur(img, k, rng=None):
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0 / k                      # horizontal linear motion
    return cv2.filter2D(img, -1, kernel, borderType=cv2.BORDER_REPLICATE)


def deg_jpeg(img, quality, rng=None):
    buf = io.BytesIO()
    Image.fromarray((img * 255.0).round().astype(np.uint8), mode='L').save(
        buf, format='JPEG', quality=int(quality))
    buf.seek(0)
    return np.asarray(Image.open(buf), dtype=np.float32) / 255.0


def deg_downsample(img, scale, rng=None):
    h, w = img.shape
    small = cv2.resize(img, (int(round(w * scale)), int(round(h * scale))),
                       interpolation=cv2.INTER_LINEAR)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def deg_contrast(img, gamma, rng=None):
    return np.clip(img, 0.0, 1.0) ** gamma


DEGRADE = {
    'Gaussian noise':     deg_gaussian_noise,
    'Motion blur':        deg_motion_blur,
    'JPEG compression':   deg_jpeg,
    'Downsampling':       deg_downsample,
    'Contrast reduction': deg_contrast,
}


def apply_degradation(img, condition, severity, rng):
    if condition == 'Clean':
        return img
    return DEGRADE[condition](img, SEVERITIES[condition][severity], rng)

### Visual sanity check

Confirms the operators behave as described before any training time is spent.

In [ ]:
import matplotlib.pyplot as plt

sample = manifest[manifest.label == 1].iloc[0]
raw = np.asarray(Image.open(sample.path).convert('L').resize((IMG_SIZE, IMG_SIZE),
                                                             Image.BILINEAR), dtype=np.float32)
base = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(1, 6, figsize=(18, 3.2))
axes[0].imshow(base, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Clean')
for ax, cond in zip(axes[1:], SEVERITIES):
    ax.imshow(apply_degradation(base, cond, 'Moderate', rng), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{cond}\n(moderate)', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Dataset and preprocessing

Order: load as float32 -> resize 224x224 bilinear -> per-image min-max to [0,1]
-> degradation (test only) -> replicate to three channels.

No augmentation, denoising, sharpening or contrast enhancement at any stage.

In [ ]:
class CXRDataset(Dataset):
    def __init__(self, df, condition='Clean', severity=None, seed=SEED):
        self.df = df.reset_index(drop=True)
        self.condition = condition
        self.severity = severity
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row.path).convert('L').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        img = np.asarray(img, dtype=np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)      # per-image min-max
        img = apply_degradation(img, self.condition, self.severity, self.rng)
        x = torch.from_numpy(np.repeat(img[None], 3, axis=0).copy())  # 3 channels
        return x, torch.tensor(row.label, dtype=torch.float32), row.image_id

## 5. Models

ImageNet-pretrained weights, classification head replaced by a single logit,
fine-tuned end to end with no layers frozen.

In [ ]:
def build_model(name):
    if name == 'ResNet-50':
        m = torchvision.models.resnet50(weights='IMAGENET1K_V1')
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == 'DenseNet-121':
        m = torchvision.models.densenet121(weights='IMAGENET1K_V1')
        m.classifier = nn.Linear(m.classifier.in_features, 1)
    elif name == 'MobileNetV2':
        m = torchvision.models.mobilenet_v2(weights='IMAGENET1K_V1')
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    else:
        raise ValueError(name)
    for p in m.parameters():          # nothing frozen
        p.requires_grad = True
    return m.to(DEVICE)


for name in MODELS:
    n = sum(p.numel() for p in build_model(name).parameters())
    print(f'{name:14s} {n/1e6:5.1f} M parameters')

## 6. Training

Binary cross-entropy, Adam at 1e-4, LR x0.1 after 5 epochs without validation-loss
improvement, batch size 16, up to 50 epochs, early stopping on validation AUC with
patience 10. No class weighting. Hyperparameters are fixed a priori and identical
across all architectures and folds - there is no search.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    ids, ys, ps = [], [], []
    for x, y, iid in loader:
        logits = model(x.to(DEVICE)).squeeze(1)
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.numpy())
        ids.extend(iid)
    return np.array(ids), np.concatenate(ys), np.concatenate(ps)


def train_one(model_name, train_df, val_df):
    set_seed(SEED)
    model = build_model(model_name)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)
    crit = nn.BCEWithLogitsLoss()

    tl = DataLoader(CXRDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    vl = DataLoader(CXRDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    best_auc, best_state, since = -np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for x, y, _ in tl:
            opt.zero_grad()
            loss = crit(model(x.to(DEVICE)).squeeze(1), y.to(DEVICE))
            loss.backward()
            opt.step()

        _, yv, pv = predict(model, vl)
        val_loss = nn.functional.binary_cross_entropy(
            torch.tensor(pv).clamp(1e-6, 1 - 1e-6), torch.tensor(yv)).item()
        val_auc = roc_auc_score(yv, pv)
        sched.step(val_loss)

        if val_auc > best_auc:
            best_auc, since = val_auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= ES_PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, best_auc, epoch

## 7. Run the folds

For each architecture and fold: train, fix the threshold on the validation subset by
Youden's J, then evaluate on the clean test fold and on all 15 degraded versions of it.

Writes `results/predictions/val_predictions.csv` and `test_predictions.csv`.

In [ ]:
def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])


val_rows, test_rows, thr_rows = [], [], []

for model_name in MODELS:
    for fold in range(1, N_FOLDS + 1):
        train_df, val_df, test_df = split_fold(manifest, fold)
        model, best_auc, n_ep = train_one(model_name, train_df, val_df)

        vl = DataLoader(CXRDataset(val_df), batch_size=BATCH_SIZE, num_workers=2)
        ids, yv, pv = predict(model, vl)
        thr = youden_threshold(yv, pv)
        thr_rows.append({'Model': model_name, 'Fold': fold, 'Threshold': thr})
        val_rows += [{'model': model_name, 'fold': fold, 'image_id': i,
                      'y_true': int(a), 'y_prob': float(b)}
                     for i, a, b in zip(ids, yv, pv)]

        for condition in CONDITIONS:
            sevs = ['-'] if condition == 'Clean' else ['Mild', 'Moderate', 'Severe']
            for sev in sevs:
                ds = CXRDataset(test_df, condition, None if sev == '-' else sev,
                                seed=SEED + fold)   # fixed noise realisation per condition
                ids, yt, pt = predict(model, DataLoader(ds, batch_size=BATCH_SIZE,
                                                        num_workers=2))
                test_rows += [{'model': model_name, 'condition': condition, 'severity': sev,
                               'fold': fold, 'image_id': i, 'y_true': int(a),
                               'y_prob': float(b)} for i, a, b in zip(ids, yt, pt)]

        print(f'{model_name:14s} fold {fold}  val AUC {best_auc:.3f}  '
              f'thr {thr:.2f}  epochs {n_ep}')

pd.DataFrame(val_rows).to_csv(OUT / 'predictions/val_predictions.csv', index=False)
pd.DataFrame(test_rows).to_csv(OUT / 'predictions/test_predictions.csv', index=False)
pd.DataFrame(thr_rows).to_csv(OUT / 'tables/thresholds.csv', index=False)
print('\nthreshold range: '
      f"{min(r['Threshold'] for r in thr_rows):.2f} to "
      f"{max(r['Threshold'] for r in thr_rows):.2f}")

## 8. Metrics

ECE uses 10 equal-width confidence bins, each weighted by its sample count.
All threshold-dependent metrics are computed **within** a fold and averaged across folds
with equal weight - folds are not pooled before computing metrics.

In [ ]:
def confusion(y, p, thr):
    pred = (p >= thr).astype(int)
    return (int(((pred == 1) & (y == 1)).sum()), int(((pred == 1) & (y == 0)).sum()),
            int(((pred == 0) & (y == 0)).sum()), int(((pred == 0) & (y == 1)).sum()))


def ece(y, p, n_bins=N_BINS, scheme='equal-width'):
    conf = np.where(p >= 0.5, p, 1.0 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    if scheme == 'equal-width':
        edges = np.linspace(0.0, 1.0, n_bins + 1)
    else:                                     # equal-mass
        edges = np.quantile(conf, np.linspace(0, 1, n_bins + 1))
        edges[0], edges[-1] = 0.0, 1.0
    idx = np.clip(np.digitize(conf, edges[1:-1], right=False), 0, n_bins - 1)
    out = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            out += (m.sum() / len(y)) * abs(correct[m].mean() - conf[m].mean())
    return float(out)


def metrics(y, p, thr):
    tp, fp, tn, fn = confusion(y, p, thr)
    return {'AUC': roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
            'Sensitivity': tp / (tp + fn) if tp + fn else np.nan,
            'Specificity': tn / (tn + fp) if tn + fp else np.nan,
            'PPV': tp / (tp + fp) if tp + fp else np.nan,
            'NPV': tn / (tn + fn) if tn + fn else np.nan,
            'ECE': ece(y, p),
            'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn, 'n': tp + fp + tn + fn}

## 9. Fold-level results and Supplementary Table S2

S2 carries the confusion-matrix counts and the threshold for every model, condition and fold -
the evidence a reader needs to reproduce Table 3 by hand.

In [ ]:
test = pd.read_csv(OUT / 'predictions/test_predictions.csv')
thresholds = {(r.Model, r.Fold): r.Threshold
              for r in pd.read_csv(OUT / 'tables/thresholds.csv').itertuples()}

rows = []
for (m, c, s, f), g in test.groupby(['model', 'condition', 'severity', 'fold']):
    thr = thresholds[(m, f)]
    rows.append({'Model': m, 'Condition': c, 'Severity': s, 'Fold': f,
                 'Threshold': round(thr, 4),
                 **metrics(g.y_true.values, g.y_prob.values, thr)})

fold_level = pd.DataFrame(rows)
cols = ['Model', 'Condition', 'Severity', 'Fold', 'Threshold', 'n',
        'TP', 'FP', 'TN', 'FN', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'AUC', 'ECE']
fold_level[cols].to_csv(OUT / 'tables/supplementary_table_s2.csv', index=False)

point = (fold_level.groupby(['Model', 'Condition', 'Severity'])
         [['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'ECE']].mean().reset_index())
point.head(12)

## 10. Paired bootstrap confidence intervals

1000 resamples, drawn once per fold and reused across every condition so clean and degraded
comparisons stay paired, stratified by TB status. Intervals are 95% percentile intervals,
unadjusted for multiplicity across the 45 model x degradation x severity combinations.

In [ ]:
rng = np.random.default_rng(SEED)
clean = test[test.condition == 'Clean']
boot_idx = {}
for (m, f), g in clean.groupby(['model', 'fold']):
    ids = g.image_id.values
    pos, neg = ids[g.y_true.values == 1], ids[g.y_true.values == 0]
    boot_idx[(m, f)] = [np.concatenate([rng.choice(pos, len(pos), replace=True),
                                        rng.choice(neg, len(neg), replace=True)])
                        for _ in range(N_BOOT)]

ci_rows = []
for (m, c, s), g in test.groupby(['model', 'condition', 'severity']):
    per_fold = {f: gg.set_index('image_id') for f, gg in g.groupby('fold')}
    thr = float(np.mean([thresholds[(m, f)] for f in per_fold]))
    draws = {k: [] for k in ['AUC', 'Sensitivity', 'Specificity', 'ECE']}
    for b in range(N_BOOT):
        yt = np.concatenate([per_fold[f].loc[boot_idx[(m, f)][b]].y_true.values
                             for f in per_fold])
        yp = np.concatenate([per_fold[f].loc[boot_idx[(m, f)][b]].y_prob.values
                             for f in per_fold])
        mm = metrics(yt, yp, thr)
        for k in draws:
            draws[k].append(mm[k])
    row = {'Model': m, 'Condition': c, 'Severity': s}
    for k, v in draws.items():
        row[f'{k}_lo'], row[f'{k}_hi'] = np.percentile(v, [2.5, 97.5])
    ci_rows.append(row)

ci = pd.DataFrame(ci_rows)
print(ci.shape)

## 11. Manuscript tables

Every table below is derived from `point` and `ci`, so they cannot disagree with each other
or with Supplementary Table S2.

In [ ]:
def fmt(v, lo, hi, d=2):
    return f'{v:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]'


S1 = point.merge(ci, on=['Model', 'Condition', 'Severity'])
clean_sens = point[point.Condition == 'Clean'].set_index('Model').Sensitivity
S1['Delta_sensitivity_pct'] = [
    np.nan if r.Condition == 'Clean' else
    100 * (r.Sensitivity - clean_sens[r.Model]) / clean_sens[r.Model] for r in S1.itertuples()]
for c, d in [('AUC', 2), ('Sensitivity', 2), ('Specificity', 2), ('ECE', 3)]:
    S1[f'{c} (95% CI)'] = [fmt(getattr(r, c), getattr(r, f'{c}_lo'), getattr(r, f'{c}_hi'), d)
                           for r in S1.itertuples()]

order_c = {c: i for i, c in enumerate(CONDITIONS)}
order_s = {s: i for i, s in enumerate(['-', 'Mild', 'Moderate', 'Severe'])}
order_m = {m: i for i, m in enumerate(MODELS)}
S1 = S1.sort_values(by=['Condition', 'Severity', 'Model'],
                    key=lambda col: col.map(order_c | order_s | order_m))

S1_out = S1[['Condition', 'Severity', 'Model', 'AUC (95% CI)', 'Sensitivity (95% CI)',
             'Specificity (95% CI)', 'ECE (95% CI)', 'Delta_sensitivity_pct']]
S1_out.to_csv(OUT / 'tables/supplementary_table_s1.csv', index=False)
S1_out.head(10)

In [ ]:
mod = point[point.Severity == 'Moderate'].copy()
mod['RelDrop'] = [100 * (r.Sensitivity - clean_sens[r.Model]) / clean_sens[r.Model]
                  for r in mod.itertuples()]

# Table 1 - clean baseline
t1 = point[point.Condition == 'Clean'].set_index('Model')[
    ['AUC', 'Sensitivity', 'Specificity', 'ECE']].round(3)
t1.to_csv(OUT / 'tables/table1_clean.csv')

# Table 3 - PPV and NPV, clean vs mean of the five moderate conditions
t3 = pd.DataFrame({
    'PPV clean': point[point.Condition == 'Clean'].set_index('Model').PPV,
    'PPV degraded': mod.groupby('Model').PPV.mean(),
    'NPV clean': point[point.Condition == 'Clean'].set_index('Model').NPV,
    'NPV degraded': mod.groupby('Model').NPV.mean()}).round(3)
t3['PPV drop %'] = (100 * (t3['PPV degraded'] - t3['PPV clean']) / t3['PPV clean']).round(1)
t3['NPV drop %'] = (100 * (t3['NPV degraded'] - t3['NPV clean']) / t3['NPV clean']).round(1)
t3.to_csv(OUT / 'tables/table3_ppv_npv.csv')

# Table 4 - relative sensitivity drops by degradation type
t4 = mod.pivot_table(index='Model', columns='Condition', values='RelDrop').round(1)
t4.to_csv(OUT / 'tables/table4_sensitivity.csv')

# Table 5 - AUC and ECE, clean vs degraded average
t5 = pd.DataFrame({
    'AUC clean': point[point.Condition == 'Clean'].set_index('Model').AUC,
    'AUC degraded': mod.groupby('Model').AUC.mean(),
    'ECE clean': point[point.Condition == 'Clean'].set_index('Model').ECE,
    'ECE degraded': mod.groupby('Model').ECE.mean()}).round(3)
t5.to_csv(OUT / 'tables/table5_auc_ece.csv')

# Table 6 - DenseNet-121 error rates, clean vs severe motion blur
def rates(model, cond, sev):
    r = point[(point.Model == model) & (point.Condition == cond) &
              (point.Severity == sev)].iloc[0]
    return {'TPR (Sensitivity)': round(r.Sensitivity, 2), 'FNR': round(1 - r.Sensitivity, 2),
            'TNR (Specificity)': round(r.Specificity, 2), 'FPR': round(1 - r.Specificity, 2)}

t6 = pd.DataFrame({'Clean': rates('DenseNet-121', 'Clean', '-'),
                   'Motion blur (k = 9, severe)': rates('DenseNet-121', 'Motion blur',
                                                        'Severe')}).T
t6.to_csv(OUT / 'tables/table6_error_rates.csv')

# Table 7 - relative decline alongside absolute sensitivity
t7 = pd.DataFrame({
    'Clean sensitivity': clean_sens,
    'Mean relative reduction (%)': mod.groupby('Model').RelDrop.mean().round(1),
    'Mean absolute sensitivity under degradation': mod.groupby('Model').Sensitivity.mean()
}).round(3)
t7.to_csv(OUT / 'tables/table7_robustness.csv')

for name, t in [('Table 1', t1), ('Table 3', t3), ('Table 4', t4),
                ('Table 5', t5), ('Table 6', t6), ('Table 7', t7)]:
    print(f'\n=== {name} ==='); print(t)

## 12. Consistency and robustness checks

These are the checks a reviewer will run. Run them before writing any number into the
manuscript.

1. Every S2 row sums to the fold size, and TP+FN matches the fold's TB-positive count.
2. Table 4's relative drops equal those implied by S1's absolute sensitivities.
3. Whether the largest sensitivity loss occurs at moderate or severe severity - state the
   severity that the headline figure in the abstract actually refers to.
4. ECE is not an artefact of the binning scheme.

In [ ]:
s2 = pd.read_csv(OUT / 'tables/supplementary_table_s2.csv')

bad_n = s2[s2.n != s2[['TP', 'FP', 'TN', 'FN']].sum(axis=1)]
print('rows where counts do not sum to n:', len(bad_n))
print('TB-positive per fold (TP+FN):', sorted((s2.TP + s2.FN).unique()))
print('test fold sizes:', sorted(s2.n.unique()))

check = mod[['Model', 'Condition', 'RelDrop']].copy()
check['from_S1'] = [S1[(S1.Model == r.Model) & (S1.Condition == r.Condition) &
                       (S1.Severity == 'Moderate')].Delta_sensitivity_pct.iloc[0]
                    for r in mod.itertuples()]
check['agree'] = np.isclose(check.RelDrop, check.from_S1, atol=0.05)
print('\nTable 4 vs S1 rows in agreement:', int(check.agree.sum()), 'of', len(check))

In [ ]:
worst = []
for m in MODELS:
    for sev in ['Mild', 'Moderate', 'Severe']:
        r = point[(point.Model == m) & (point.Condition == 'Motion blur') &
                  (point.Severity == sev)]
        if len(r):
            drop = 100 * (r.Sensitivity.iloc[0] - clean_sens[m]) / clean_sens[m]
            worst.append({'Model': m, 'Severity': sev, 'Sensitivity drop %': round(drop, 1)})
worst = pd.DataFrame(worst)
print('Motion blur sensitivity loss by severity:')
print(worst.pivot(index='Model', columns='Severity', values='Sensitivity drop %'))
print('\nLargest single loss:', worst.loc[worst['Sensitivity drop %'].idxmin()].to_dict())
print('-> the abstract must attribute this figure to the severity shown above.')

In [ ]:
rows = []
for m in MODELS:
    g_clean = test[(test.model == m) & (test.condition == 'Clean')]
    g_mod = test[(test.model == m) & (test.severity == 'Moderate')]
    for label, bins, scheme in [('10 equal-width', 10, 'equal-width'),
                                ('10 equal-mass', 10, 'equal-mass'),
                                ('5 equal-width', 5, 'equal-width'),
                                ('15 equal-width', 15, 'equal-width')]:
        c = ece(g_clean.y_true.values, g_clean.y_prob.values, bins, scheme)
        d = ece(g_mod.y_true.values, g_mod.y_prob.values, bins, scheme)
        rows.append({'Model': m, 'Scheme': label, 'ECE clean': round(c, 3),
                     'ECE degraded': round(d, 3), 'Increase x': round(d / c, 2) if c else np.nan})

binchk = pd.DataFrame(rows)
binchk.to_csv(OUT / 'tables/ece_bin_sensitivity.csv', index=False)
print(binchk.pivot(index='Model', columns='Scheme', values='Increase x'))
print('\nThe manuscript claims the direction and approximate magnitude are unchanged;')
print('confirm that from the row above before keeping that sentence.')

## 13. Exploratory feature-distance analysis

Mean L2 distance between penultimate-layer features of clean and degraded images, for
TB-positive cases, DenseNet-121 under moderate motion blur. Reported as the mean across
cases with the standard deviation across folds.

The within-class distance between pairs of clean images is computed alongside it, because
L2 in this space is scale-dependent and the raw value is not interpretable on its own.

In [ ]:
# Requires the trained DenseNet-121 fold models. If you re-ran Section 7 in this session,
# retrain or reload per fold before running this cell.

@torch.no_grad()
def penultimate(model, loader):
    feats = []
    backbone = nn.Sequential(model.features, nn.ReLU(inplace=False),
                             nn.AdaptiveAvgPool2d(1), nn.Flatten())
    backbone.eval()
    for x, _, _ in loader:
        feats.append(backbone(x.to(DEVICE)).cpu().numpy())
    return np.concatenate(feats)


def feature_distance(fold_models, manifest):
    per_fold, within = [], []
    for fold, model in fold_models.items():
        _, _, test_df = split_fold(manifest, fold)
        pos = test_df[test_df.label == 1]
        fc = penultimate(model, DataLoader(CXRDataset(pos), batch_size=BATCH_SIZE))
        fd = penultimate(model, DataLoader(CXRDataset(pos, 'Motion blur', 'Moderate'),
                                          batch_size=BATCH_SIZE))
        per_fold.append(np.linalg.norm(fc - fd, axis=1).mean())
        d = np.linalg.norm(fc[:, None, :] - fc[None, :, :], axis=-1)
        within.append(d[np.triu_indices(len(fc), k=1)].mean())
    return (float(np.mean(per_fold)), float(np.std(per_fold)),
            float(np.mean(within)))


# m, sd, ref = feature_distance(densenet_fold_models, manifest)
# print(f'clean-to-degraded L2: {m:.1f} +/- {sd:.1f}')
# print(f'within-class clean L2: {ref:.1f}  ->  shift is {100*m/ref:.0f}% of within-class spread')

---

## Outputs

| File | Contents |
|---|---|
| `results/folds.csv` | fold assignment for all 800 images |
| `results/predictions/val_predictions.csv` | validation predictions, used only for thresholds |
| `results/predictions/test_predictions.csv` | every test prediction, clean and degraded |
| `results/tables/thresholds.csv` | per-fold Youden threshold |
| `results/tables/supplementary_table_s1.csv` | absolute metrics, all 16 condition-severity rows |
| `results/tables/supplementary_table_s2.csv` | fold-level confusion-matrix counts |
| `results/tables/table{1,3,4,5,6,7}_*.csv` | manuscript tables |
| `results/tables/ece_bin_sensitivity.csv` | ECE under alternative binning schemes |

## Notes on scope

Degradations are applied to images already resized to 224x224, so the parameters describe
perturbations in the network input space rather than calibrated measurements at the detector.
Downsampling and motion blur are both low-pass operations and are not fully independent.
Each noise condition uses a single fixed realisation. No quantitative image-quality measures
are computed on the degraded images.